## 2.4 Machine Learning



In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV

# Pipeline (referencia para secciones posteriores)
from sklearn.pipeline import Pipeline

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# Métricas
from sklearn.metrics import accuracy_score

# Para guardar el modelo
import pickle

# Intentar importar modelos opcionales
XGB_AVAILABLE = True
LGBM_AVAILABLE = True

try:
    from xgboost import XGBClassifier
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBM_AVAILABLE = False

print(f"XGBoost disponible: {XGB_AVAILABLE}")
print(f"LightGBM disponible: {LGBM_AVAILABLE}")

XGBoost disponible: True
LightGBM disponible: False


### Carga de datos



In [9]:
from pathlib import Path

candidatos = [
    Path('./data/titanic_procesado.csv'),
    Path('./data/titanic_feature_engineering.csv'),
]

ruta_datos = next((ruta for ruta in candidatos if ruta.exists()), None)
if ruta_datos is None:
    raise FileNotFoundError(
        "No se encontró un archivo procesado. Se buscó en: "
        + ", ".join(str(r) for r in candidatos)
    )

df = pd.read_csv(ruta_datos)
print(f"Archivo cargado: {ruta_datos}")
print(f"Forma del dataset: {df.shape}")
df.head()

Archivo cargado: data/titanic_feature_engineering.csv
Forma del dataset: (891, 8)


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,1,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,0,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


### División de datos


In [3]:
X = df.drop(['Survived'], axis=1)
y = df['Survived']

print('Primeras filas de X:')
display(X.head())

print('Primeras filas de y:')
display(y.head())

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.values
y_train = y_train.values
X_test = X_test.values
y_test = y_test.values

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")

Primeras filas de X:


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1.0,1.0,0.433152,0.125,0.0,0.368146,1.0
1,0.0,0.0,0.579431,0.125,0.0,0.615097,0.0
2,1.0,0.0,0.462346,0.000,0.0,0.438286,1.0
3,0.0,0.0,0.563806,0.125,0.0,0.595112,1.0
4,1.0,1.0,0.563806,0.000,0.0,0.448347,1.0


Primeras filas de y:


0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

X_train: (712, 7)
X_test: (179, 7)


### Selección de modelo con GridSearchCV



In [4]:
# Definir los modelos y sus respectivos hiperparámetros para GridSearch
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga'],
            'max_iter': [100, 500, 1000]
        }
    },
    'Clasificador de Vectores de Soporte': {
        'modelo': SVC(),
        'parametros': {
            'kernel': ['linear', 'poly', 'rbf', 'sigmoid'],
            'C': [0.1, 1, 10]
        }
    },
    'Clasificador de Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {
            'splitter': ['best', 'random'],
            'max_depth': [None, 1, 2, 3, 4]
        }
    },
    'Clasificador de Bosques Aleatorios': {
        'modelo': RandomForestClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [None, 1, 2, 3, 4],
            'max_features': ['sqrt', 'log2', None]
        }
    },
    'Clasificador de Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [1, 2, 3, 4]
        }
    },
    'Clasificador AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {
            'n_estimators': [10, 100]
        }
    },
    'Clasificador K-Nearest Neighbors': {
        'modelo': KNeighborsClassifier(),
        'parametros': {
            'n_neighbors': [3, 5, 7]
        }
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'Clasificador Naive Bayes': {
        'modelo': BernoulliNB(),
        'parametros': {
            'alpha': [0.1, 1.0, 10.0]
        }
    }
}

if XGB_AVAILABLE:
    modelos['Clasificador XGBoost'] = {
        'modelo': XGBClassifier(eval_metric='logloss', random_state=42),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [1, 2, 3],
        }
    }

if LGBM_AVAILABLE:
    modelos['Clasificador LGBM'] = {
        'modelo': LGBMClassifier(random_state=42, verbose=-1),
        'parametros': {
            'n_estimators': [10, 100],
            'max_depth': [-1, 1, 2, 3],
            'learning_rate': [0.1, 0.2, 0.3],
            'verbose': [-1]
        }
    }

# Inicializar variables para almacenar resultados
puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo = None
estimadores = {}
modelos_con_error = []

# Iterar sobre cada modelo y sus hiperparámetros
for nombre, info_modelo in modelos.items():
    try:
        grid_search = GridSearchCV(
            estimator=info_modelo['modelo'],
            param_grid=info_modelo['parametros'],
            cv=5,
            scoring='accuracy',
            verbose=0,
            n_jobs=-1,
        )

        grid_search.fit(X_train, y_train)
        y_pred = grid_search.predict(X_test)
        precision = accuracy_score(y_test, y_pred)

        puntajes_modelos.append({
            'Modelo': nombre,
            'Precisión': precision
        })

        estimadores[nombre] = grid_search.best_estimator_

        if precision > mejor_precision:
            mejor_modelo = nombre
            mejor_precision = precision
            mejor_estimador = grid_search.best_estimator_

    except Exception as e:
        modelos_con_error.append((nombre, str(e)))

metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)

print('Rendimiento de los modelos de clasificación')
display(metricas.round(2))

print('---------------------------------------------------')
print('MEJOR MODELO DE CLASIFICACIÓN')
print(f'Modelo: {mejor_modelo}')
print(f'Precisión: {mejor_precision:.2f}')

if modelos_con_error:
    print('\nModelos omitidos por error:')
    for nombre, err in modelos_con_error:
        print(f'- {nombre}: {err[:180]}...')

/home/alex/Hybridge/IA/Lenguaje supervisado/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/home/alex/Hybridge/IA/Lenguaje supervisado/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
/home/alex/Hybridge/IA/Lenguaje supervisado/aprendizaje_supervisado/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set 

Rendimiento de los modelos de clasificación


,Modelo,Precisión
6,Clasificador K-Nearest Neighbors,0.83
4,Clasificador de Gradient Boosting,0.82
9,Clasificador XGBoost,0.82
1,Clasificador de Vectores de Soporte,0.80
2,Clasificador de Árbol de Decisión,0.80
0,Regresión Logística,0.79
5,Clasificador AdaBoost,0.79
3,Clasificador de Bosques Aleatorios,0.79
8,Clasificador Naive Bayes,0.79
7,GaussianNB,0.77


---------------------------------------------------
MEJOR MODELO DE CLASIFICACIÓN
Modelo: Clasificador K-Nearest Neighbors
Precisión: 0.83


### Revisión rápida: parámetros vs hiperparámetros


### Sin GridSearch (modelo individual)

Ejemplo rápido con regresión logística usando valores por defecto:

In [5]:
modelo_logistico = LogisticRegression()
modelo_logistico.fit(X_train, y_train)

y_pred_lr = modelo_logistico.predict(X_test)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

print(f'Precisión (Regresión Logística por defecto): {accuracy_lr:.2f}')

Precisión (Regresión Logística por defecto): 0.80


### Inferencia

Usamos `mejor_estimador` para predecir sobre datos nuevos. El arreglo debe respetar exactamente el mismo orden de variables que `X_train`.

In [6]:
if mejor_estimador is None:
    raise RuntimeError('No hay un mejor estimador disponible. Revisa si hubo errores en el entrenamiento.')

# Ejemplo: usar la primera fila de entrenamiento como nueva observación
nuevos_datos = X_train[0].reshape(1, -1)
prediccion = mejor_estimador.predict(nuevos_datos)

print('Datos de entrada de ejemplo:')
print(nuevos_datos)
print(f'Predicción del mejor modelo: {prediccion[0]}')

Datos de entrada de ejemplo:
[[0.         1.         0.61581699 0.         0.         0.55559921
  1.        ]]
Predicción del mejor modelo: 0


### Guardar el modelo

Guardamos el mejor estimador para usarlo después en producción.

In [7]:
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print('Modelo guardado en: modelo.pkl')

Modelo guardado en: modelo.pkl
